# UNSW-NB15 Exploratory Data Analysis

This notebook performs a clean first-pass exploratory analysis for the processed UNSW-NB15 network intrusion dataset. It loads the processed CSV configured in `src/config/config.yaml`, summarizes the dataset, analyzes the target variable, creates visualizations, and saves reusable EDA artifacts under `reports/`.

## 1. Setup

In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import yaml

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

CONFIG_PATH = PROJECT_ROOT / "src" / "config" / "config.yaml"
REPORTS_DIR = PROJECT_ROOT / "reports"
IMAGES_DIR = REPORTS_DIR / "images"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
IMAGES_DIR.mkdir(parents=True, exist_ok=True)

plt.style.use("ggplot")
pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 140)

## 2. Load Processed Dataset

In [ ]:
with CONFIG_PATH.open("r", encoding="utf-8") as config_file:
    config = yaml.safe_load(config_file)

processed_dataset_path = PROJECT_ROOT / config["data"]["processed_dataset_path"]
target_column = config["data"].get("target_column", "label")

if not processed_dataset_path.exists():
    raise FileNotFoundError(
        f"Processed dataset not found at {processed_dataset_path}. "
        "Run the data ingestion pipeline before running this notebook."
    )

df = pd.read_csv(processed_dataset_path)
df.head()

## 3. Dataset Overview

In [ ]:
print(f"Shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns")
print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

memory_mb = df.memory_usage(deep=True).sum() / (1024 ** 2)
print(f"\nMemory usage: {memory_mb:.2f} MB")

## 4. Target Variable Analysis

In [ ]:
if target_column not in df.columns:
    raise ValueError(f"Target column '{target_column}' was not found in the dataset.")

target_counts = df[target_column].value_counts(dropna=False)
target_percentages = df[target_column].value_counts(normalize=True, dropna=False).mul(100).round(2)

target_summary = pd.DataFrame(
    {
        "count": target_counts,
        "percentage": target_percentages,
    }
)

target_summary

In [ ]:
fig, ax = plt.subplots(figsize=(10, 5))
target_counts.plot(kind="bar", ax=ax, color="#2563eb")
ax.set_title("Target Class Distribution")
ax.set_xlabel(target_column)
ax.set_ylabel("Record Count")
ax.tick_params(axis="x", rotation=45)
fig.tight_layout()
fig.savefig(IMAGES_DIR / "target_class_distribution.png", dpi=150)
plt.show()

## 5. Column Type Identification

In [ ]:
numerical_columns = df.select_dtypes(include="number").columns.tolist()
categorical_columns = df.select_dtypes(exclude="number").columns.tolist()

print(f"Numerical columns ({len(numerical_columns)}):")
print(numerical_columns)

print(f"\nCategorical columns ({len(categorical_columns)}):")
print(categorical_columns)

## 6. Missing Values and Duplicate Rows

In [ ]:
missing_values = df.isna().sum().sort_values(ascending=False)
missing_percentages = (df.isna().mean() * 100).round(2).sort_values(ascending=False)
missing_summary = pd.DataFrame(
    {
        "missing_count": missing_values,
        "missing_percentage": missing_percentages,
    }
)

duplicate_rows = int(df.duplicated().sum())
print(f"Duplicate rows: {duplicate_rows:,}")

missing_summary[missing_summary["missing_count"] > 0].head(25)

## 7. Descriptive Statistics

In [ ]:
feature_statistics = df.describe(include="all").transpose()
feature_statistics.to_csv(REPORTS_DIR / "feature_statistics.csv")
feature_statistics.head(20)

## 8. Correlation Matrix

In [ ]:
if numerical_columns:
    correlation_matrix = df[numerical_columns].corr()
else:
    correlation_matrix = pd.DataFrame()

correlation_matrix.to_csv(REPORTS_DIR / "correlation_matrix.csv")
correlation_matrix

## 9. Histograms

In [ ]:
if numerical_columns:
    axes = df[numerical_columns].hist(figsize=(16, 12), bins=30, color="#0f766e")
    for ax in axes.flatten():
        ax.set_xlabel(ax.get_title())
        ax.set_ylabel("Frequency")
    plt.suptitle("Numerical Feature Histograms", y=1.02)
    plt.tight_layout()
    plt.savefig(IMAGES_DIR / "numerical_histograms.png", dpi=150, bbox_inches="tight")
    plt.show()
else:
    print("No numerical columns available for histograms.")

## 10. Boxplots

In [ ]:
if numerical_columns:
    boxplot_columns = numerical_columns[:20]
    fig, ax = plt.subplots(figsize=(16, 7))
    df[boxplot_columns].plot(kind="box", ax=ax, vert=False)
    ax.set_title("Boxplots for Numerical Features")
    ax.set_xlabel("Value")
    fig.tight_layout()
    fig.savefig(IMAGES_DIR / "numerical_boxplots.png", dpi=150)
    plt.show()
else:
    print("No numerical columns available for boxplots.")

## 11. Correlation Heatmap

In [ ]:
if not correlation_matrix.empty:
    fig, ax = plt.subplots(figsize=(14, 10))
    heatmap = ax.imshow(correlation_matrix, cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_title("Correlation Heatmap")
    ax.set_xticks(range(len(correlation_matrix.columns)))
    ax.set_xticklabels(correlation_matrix.columns, rotation=90)
    ax.set_yticks(range(len(correlation_matrix.index)))
    ax.set_yticklabels(correlation_matrix.index)
    fig.colorbar(heatmap, ax=ax, fraction=0.046, pad=0.04)
    fig.tight_layout()
    fig.savefig(IMAGES_DIR / "correlation_heatmap.png", dpi=150)
    plt.show()
else:
    print("No numerical columns available for a correlation heatmap.")

## 12. Save EDA Summary

In [ ]:
top_missing = missing_summary[missing_summary["missing_count"] > 0].head(10)

summary_lines = [
    "# EDA Summary - UNSW-NB15",
    "",
    f"- Dataset path: `{processed_dataset_path}`",
    f"- Shape: {df.shape[0]:,} rows x {df.shape[1]:,} columns",
    f"- Memory usage: {memory_mb:.2f} MB",
    f"- Target column: `{target_column}`",
    f"- Numerical columns: {len(numerical_columns)}",
    f"- Categorical columns: {len(categorical_columns)}",
    f"- Duplicate rows: {duplicate_rows:,}",
    f"- Columns with missing values: {(missing_summary['missing_count'] > 0).sum()}",
    "",
    "## Target Distribution",
    "",
    "```",
    target_summary.to_string(),
    "```",
    "",
    "## Top Missing Value Columns",
    "",
    "```",
    top_missing.to_string() if not top_missing.empty else "No missing values found.",
    "```",
    "",
    "## Saved Artifacts",
    "",
    "- `reports/feature_statistics.csv`",
    "- `reports/correlation_matrix.csv`",
    "- `reports/eda_summary.md`",
    "- `reports/images/target_class_distribution.png`",
    "- `reports/images/numerical_histograms.png`",
    "- `reports/images/numerical_boxplots.png`",
    "- `reports/images/correlation_heatmap.png`",
]

summary_path = REPORTS_DIR / "eda_summary.md"
summary_path.write_text("\n".join(summary_lines), encoding="utf-8")

print(f"Saved feature statistics to: {REPORTS_DIR / 'feature_statistics.csv'}")
print(f"Saved correlation matrix to: {REPORTS_DIR / 'correlation_matrix.csv'}")
print(f"Saved EDA summary to: {summary_path}")
print(f"Saved plots to: {IMAGES_DIR}")